# 05 — Predicción de facturación diaria (notebook definitivo)

## Objetivo

Modelo de referencia para predecir la **facturación diaria** del restaurante, construido
a partir de lo aprendido en `04_1_facturacion_diaria.ipynb`, `04_2_prediccion_facturacion_diaria.ipynb`
y `notebook_prediccion_facturacion_restaurante (1).ipynb`. Este notebook se queda con las mejores
decisiones de cada uno y corrige sus puntos débiles:

- **Fuente de datos**: tabla maestra de **Gold** (`data/gold/tabla_maestra_diaria.parquet`), que ya
  integra tickets, festivos, eventos y meteorología a nivel diario. Se enriquece con una fuente de
  **Silver** (`reservas_silver.parquet`) para construir una variable que ningún notebook anterior
  explotaba bien: **reservas ya confirmadas con antelación**, calculada sin fuga de información.
- **Control de leakage estricto**: se excluyen todas las variables que solo se conocen al terminar
  el día (`num_tickets`, `ticket_medio`, reservas/comensales completados, tasas de no-show/cancelación...).
- **Horizonte de predicción configurable** (`HORIZON`): por defecto se predice la facturación de
  **mañana** usando todo lo conocido hasta hoy. Todas las variables históricas (lags, medias móviles,
  reservas anticipadas, festivos adyacentes) respetan ese corte temporal.
- **5 modelos**: Ridge, Random Forest, Extra Trees, HistGradientBoosting y XGBoost — mezcla de lineal,
  bagging y boosting.
- **Ajuste de hiperparámetros real**: `GridSearchCV` / `RandomizedSearchCV` con `TimeSeriesSplit`,
  nunca con folds aleatorios (evitaría entrenar con información futura).
- **Validación honesta**: selección de modelo únicamente con el 80% de entrenamiento; el 20% final
  (cronológicamente el más reciente) se reserva como test y solo se evalúa una vez, al final.
- **Comparación contra baselines ingenuos** (día anterior y mismo día de la semana anterior): un
  modelo de ML solo es útil si supera claramente estas referencias simples.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
GOLD_DIR = DATA_DIR / 'gold'
SILVER_SNAP = DATA_DIR / 'silver' / 'snapshots'

RESULTS_DIR = PROJECT_ROOT / 'results' / 'forecasting_final'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures' / 'forecasting_final'
MODELS_DIR = PROJECT_ROOT / 'results' / 'models'
for p in (RESULTS_DIR, FIGURES_DIR, MODELS_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

print('Project root:', PROJECT_ROOT)
print('Gold dir     :', GOLD_DIR)
print('Silver snaps :', SILVER_SNAP)


## 1. Horizonte de predicción

`HORIZON` = número mínimo de días de antelación con los que se predice.

- `HORIZON = 1` → predecir la facturación de **mañana** con todo lo conocido hasta **hoy**
  (uso típico: planificación diaria de compras/personal). Es el valor que maximiza la información
  disponible y por tanto el que da mejor precisión — se usa por defecto.
- Valores mayores (`HORIZON = 3, 7...`) simulan que la predicción se necesita con más antelación
  (p. ej. para pedidos a proveedores con más lead time). Todas las variables de historial se
  recalculan automáticamente respetando el corte elegido.


In [ ]:
HORIZON = 1  # días de antelación de la predicción
print(f'HORIZON = {HORIZON} día(s) de antelación')


## 2. Carga de datos — tabla maestra (Gold) + reservas y festivos (Silver)

In [ ]:
gold = pd.read_parquet(GOLD_DIR / 'tabla_maestra_diaria.parquet')
gold['fecha'] = pd.to_datetime(gold['fecha'])
gold = gold.sort_values('fecha').drop_duplicates('fecha', keep='last').reset_index(drop=True)

reservas = pd.read_parquet(SILVER_SNAP / 'reservas_silver.parquet')
reservas['reservation_date'] = pd.to_datetime(reservas['reservation_date'])
reservas['created_date'] = pd.to_datetime(reservas['created_date'])

festivos_silver = pd.read_parquet(SILVER_SNAP / 'festivos_silver.parquet')
festivos_silver['fecha'] = pd.to_datetime(festivos_silver['fecha']).dt.normalize()

print('Tabla maestra (Gold):', gold.shape, '|', gold['fecha'].min().date(), '->', gold['fecha'].max().date())
print('Reservas (Silver)   :', reservas.shape)
print('Festivos (Silver)   :', festivos_silver.shape)

df = gold.copy()


## 3. Reservas confirmadas con antelación (feature nueva, sin leakage)

La tabla Gold trae `n_reservas_total`, pero mezcla reservas hechas el mismo día (walk-ins,
confirmaciones de última hora) con reservas antiguas — por eso se excluye como variable de leakage.

Aquí construimos algo distinto y sí utilizable: para cada fecha objetivo, cuántas reservas
**ya existían** con al menos `HORIZON` días de antelación (usando `created_date` vs `reservation_date`
de Silver). Esa información sí está disponible en el momento de predecir.

In [ ]:
reservas_validas = reservas.dropna(subset=['reservation_date', 'created_date']).copy()
reservas_validas['antelacion_dias'] = (
    reservas_validas['reservation_date'] - reservas_validas['created_date']
).dt.days

conocidas = reservas_validas[reservas_validas['antelacion_dias'] >= HORIZON]

reservas_agg = (
    conocidas
    .groupby('reservation_date')
    .agg(
        reservas_anticipadas=('reference_code', 'count'),
        comensales_anticipados=('people', 'sum'),
        grupos_grandes_anticipados=('es_grupo_grande', 'sum'),
        antelacion_media_dias=('antelacion_dias', 'mean'),
    )
    .reset_index()
    .rename(columns={'reservation_date': 'fecha'})
)

df = df.merge(reservas_agg, on='fecha', how='left')
for c in ['reservas_anticipadas', 'comensales_anticipados', 'grupos_grandes_anticipados']:
    df[c] = df[c].fillna(0)

print(f"Días con al menos 1 reserva anticipada (>= {HORIZON}d): "
      f"{(df['reservas_anticipadas'] > 0).sum()} de {len(df)}")
df[['fecha', 'reservas_anticipadas', 'comensales_anticipados',
    'grupos_grandes_anticipados', 'antelacion_media_dias']].describe().round(2)


## 4. Variables de calendario adicionales

Añadimos codificación cíclica (para que el modelo entienda que el 31 de diciembre está "cerca"
del 1 de enero) y variables de festivo adyacente, calculadas por **fecha real** contra el
calendario completo de festivos de Silver — no por posición de fila, que sería incorrecto porque
el restaurante cierra la mayoría de los lunes y las filas no son días consecutivos.

In [ ]:
d = df['fecha']

df['anio'] = d.dt.year
df['semana_anio'] = d.dt.isocalendar().week.astype(int)
df['trimestre'] = d.dt.quarter
df['dia_anio'] = d.dt.dayofyear

df['sin_dia_anio'] = np.sin(2 * np.pi * df['dia_anio'] / 365.25)
df['cos_dia_anio'] = np.cos(2 * np.pi * df['dia_anio'] / 365.25)
df['sin_dia_semana'] = np.sin(2 * np.pi * df['num_dia'] / 7)
df['cos_dia_semana'] = np.cos(2 * np.pi * df['num_dia'] / 7)

fechas_festivas = set(festivos_silver['fecha'])
df['festivo_manana'] = (d + pd.Timedelta(days=1)).dt.normalize().isin(fechas_festivas).astype(int)
df['festivo_ayer'] = (d - pd.Timedelta(days=1)).dt.normalize().isin(fechas_festivas).astype(int)
df['es_puente'] = (
    ((d.dt.weekday == 0) & (df['festivo_manana'] == 1)) |   # lunes antes de festivo el martes
    ((d.dt.weekday == 4) & (df['festivo_ayer'] == 1))        # viernes tras festivo el jueves
).astype(int)

print('Variables de calendario creadas.')


## 5. Variables meteorológicas derivadas

In [ ]:
df['rango_temperatura'] = df['temperature_max'] - df['temperature_min']
df['es_dia_lluvioso'] = (df['precipitation_mm'].fillna(0) > 0).astype(int)
df['es_dia_muy_caluroso'] = (df['temperature_max'] >= 30).astype(int)
df['es_dia_muy_frio'] = (df['temperature_min'] <= 5).astype(int)


## 6. Historial de facturación (lags y medias móviles) — respetando `HORIZON`

`base` es la última facturación conocida en el momento de predecir (para `HORIZON=1`, es la de
ayer). Todos los lags y ventanas móviles se calculan a partir de ese corte, así que nunca usan
información posterior a lo que realmente se sabría en producción.

In [ ]:
base = df['facturacion'].shift(HORIZON)

for lag in [1, 3, 7, 14, 21, 28]:
    df[f'facturacion_{lag}d_antes'] = df['facturacion'].shift(HORIZON + lag - 1)

for w in [3, 7, 14, 28, 90]:
    df[f'facturacion_media_{w}d'] = base.rolling(w, min_periods=max(2, w // 3)).mean()
    if w in (7, 14, 28):
        df[f'facturacion_std_{w}d'] = base.rolling(w, min_periods=max(2, w // 3)).std()
        df[f'facturacion_total_{w}d'] = base.rolling(w, min_periods=max(2, w // 3)).sum()

# Momentum: ¿la media reciente está por encima o por debajo de la media larga?
df['facturacion_tendencia_7_28'] = df['facturacion_media_7d'] - df['facturacion_media_28d']

print('Variables de historial creadas.')


## 7. Selección de variables y control estricto de leakage

Se excluyen `facturacion` (objetivo) y todas las variables que son **resultado** del propio día
(número de tickets, ticket medio, reservas/comensales ya completados, tasas de no-show y
cancelación). Todo lo demás —calendario, meteorología, festivos/eventos, historial desplazado y
reservas anticipadas— se conoce antes de que empiece el día objetivo.

In [ ]:
target = 'facturacion'

leakage_cols = [
    'facturacion', 'num_tickets', 'ticket_medio', 'ticket_mediano',
    'n_reservas_cancelada', 'n_reservas_completada', 'n_reservas_no_show',
    'comensales_cancelada', 'comensales_completada', 'comensales_no_show',
    'n_reservas_total', 'tasa_no_show', 'tasa_cancelacion',
]
id_cols = ['fecha']

feature_cols = [c for c in df.columns if c not in leakage_cols + id_cols]

X = df[feature_cols].copy()
y = df[target].copy()
fechas = df['fecha'].copy()

print(f'Variables utilizadas ({len(feature_cols)}):')
for c in feature_cols:
    print(' -', c)


## 8. Filas utilizables

Se descartan solo las primeras filas que no tienen ningún histórico de facturación disponible
(imposibles de predecir de forma realista). El resto de nulos (colas de ventanas de 28/90 días,
festivo_nombre, variables de evento) los gestiona el `SimpleImputer` dentro de cada pipeline.

In [ ]:
before = len(X)
mask = X['facturacion_1d_antes'].notna()

X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)
fechas = fechas.loc[mask].reset_index(drop=True)
df_model = df.loc[mask].reset_index(drop=True)

print(f'Filas descartadas por falta de histórico mínimo: {before - len(X)}')
print(f'Filas utilizables: {len(X)}')

print('\nNulos restantes por variable (top 10):')
display(X.isna().sum().sort_values(ascending=False).head(10).to_frame('nulos'))


## 9. Train / test cronológico (80% / 20%)

El 20% final —el más reciente— nunca se usa para elegir modelo ni ajustar hiperparámetros.

In [ ]:
split = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split].copy(), X.iloc[split:].copy()
y_train, y_test = y.iloc[:split].copy(), y.iloc[split:].copy()
fechas_train, fechas_test = fechas.iloc[:split], fechas.iloc[split:]

print('TRAIN:', fechas_train.min().date(), '->', fechas_train.max().date(), f'({len(X_train)} días)')
print('TEST :', fechas_test.min().date(), '->', fechas_test.max().date(), f'({len(X_test)} días)')


## 10. Preprocesamiento

Dos ramas dentro de `ColumnTransformer`: imputación + escalado para el modelo lineal (Ridge),
e imputación sin escalar para los modelos de árboles (no lo necesitan y escalar no aporta nada).
Al vivir dentro de un `Pipeline`, cada fold de validación cruzada ajusta el preprocesado solo con
sus propios datos de entrenamiento — no hay fuga entre folds.

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

numeric_linear = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

numeric_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

preprocessor_linear = ColumnTransformer([
    ('num', numeric_linear, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

preprocessor_tree = ColumnTransformer([
    ('num', numeric_tree, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

print(f'Numéricas: {len(numeric_cols)} | Categóricas: {len(categorical_cols)}')


## 11. Cinco modelos + búsqueda de hiperparámetros

Ridge (lineal regularizado), Random Forest y Extra Trees (bagging), HistGradientBoosting y
XGBoost (boosting). Cada modelo se ajusta con `GridSearchCV`/`RandomizedSearchCV` usando
**`TimeSeriesSplit`** como esquema de validación — nunca folds aleatorios, para no entrenar con
información temporalmente posterior a la que se valida. El criterio de selección es el MAE
(interpretable en euros).

In [ ]:
tscv_search = TimeSeriesSplit(n_splits=5)

base_estimators = {
    'Ridge': (Ridge(random_state=RANDOM_STATE), preprocessor_linear),
    'Random Forest': (RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1), preprocessor_tree),
    'Extra Trees': (ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1), preprocessor_tree),
    'HistGradientBoosting': (HistGradientBoostingRegressor(random_state=RANDOM_STATE), preprocessor_tree),
    'XGBoost': (XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, objective='reg:squarederror'), preprocessor_tree),
}

param_grids = {
    'Ridge': {
        'model__alpha': [0.1, 0.3, 1, 3, 10, 30, 100, 300],
    },
    'Random Forest': {
        'model__n_estimators': [200, 400, 600],
        'model__max_depth': [None, 4, 6, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': [0.5, 0.7, 1.0],
    },
    'Extra Trees': {
        'model__n_estimators': [200, 400, 600],
        'model__max_depth': [None, 4, 6, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': [0.5, 0.7, 1.0],
    },
    'HistGradientBoosting': {
        'model__max_iter': [100, 200, 300],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
        'model__max_leaf_nodes': [7, 15, 31],
        'model__l2_regularization': [0.0, 0.5, 1.0, 2.0],
        'model__max_depth': [None, 3, 5],
    },
    'XGBoost': {
        'model__n_estimators': [200, 400, 600],
        'model__max_depth': [2, 3, 4, 6],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
        'model__subsample': [0.6, 0.8, 1.0],
        'model__colsample_bytree': [0.6, 0.8, 1.0],
        'model__reg_alpha': [0, 0.1, 0.5],
        'model__reg_lambda': [1, 2, 5],
    },
}

# Ridge tiene un espacio pequeño de un solo parámetro -> búsqueda exhaustiva.
# El resto usa búsqueda aleatoria (espacio demasiado grande para GridSearch exhaustivo).
exhaustive_models = {'Ridge'}
N_ITER_RANDOM = 25

best_estimators = {}
search_summary = []

for name, (estimator, preprocessor) in base_estimators.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    grid = param_grids[name]

    if name in exhaustive_models:
        search = GridSearchCV(
            pipe, grid, cv=tscv_search,
            scoring='neg_mean_absolute_error', n_jobs=-1,
        )
    else:
        search = RandomizedSearchCV(
            pipe, grid, n_iter=N_ITER_RANDOM, cv=tscv_search,
            scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1,
        )

    search.fit(X_train, y_train)
    best_estimators[name] = search.best_estimator_

    search_summary.append({
        'modelo': name,
        'MAE_CV_busqueda': -search.best_score_,
        'mejores_parametros': str(search.best_params_),
    })

    print(f'{name:<22} MAE_CV = {-search.best_score_:8.2f} €   params: {search.best_params_}')

search_summary_df = pd.DataFrame(search_summary).sort_values('MAE_CV_busqueda').reset_index(drop=True)


## 12. Resultado de la búsqueda de hiperparámetros

In [ ]:
display(search_summary_df)

## 13. Validación cruzada temporal fold a fold (modelos ya afinados)

No basta con mirar la media: comprobamos cómo se comporta cada modelo, ya con sus mejores
hiperparámetros, en cada periodo temporal del entrenamiento.

In [ ]:
def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    m = denom != 0
    return np.mean(2 * np.abs(y_pred[m] - y_true[m]) / denom[m]) * 100

tscv = TimeSeriesSplit(n_splits=5)
fold_rows = []

for name, pipe in best_estimators.items():
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train), start=1):
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = pipe.predict(X_train.iloc[va_idx])

        fold_rows.append({
            'modelo': name,
            'fold': fold,
            'fecha_inicio_val': fechas_train.iloc[va_idx[0]],
            'fecha_fin_val': fechas_train.iloc[va_idx[-1]],
            'MAE': mean_absolute_error(y_train.iloc[va_idx], pred),
            'RMSE': mean_squared_error(y_train.iloc[va_idx], pred) ** 0.5,
            'R2': r2_score(y_train.iloc[va_idx], pred),
            'sMAPE_%': smape(y_train.iloc[va_idx].values, pred),
        })

fold_results = pd.DataFrame(fold_rows)

cv_summary = (
    fold_results
    .groupby('modelo')
    .agg(
        MAE_mean=('MAE', 'mean'), MAE_std=('MAE', 'std'),
        RMSE_mean=('RMSE', 'mean'), R2_mean=('R2', 'mean'),
        sMAPE_mean=('sMAPE_%', 'mean'),
    )
    .sort_values('MAE_mean')
)

display(cv_summary)


## 14. Entrenamiento final y evaluación en el test futuro (nunca visto)

Cada modelo, con sus hiperparámetros ya elegidos, se reentrena con **todo** el 80% de train y se
evalúa **una sola vez** sobre el 20% final.

In [ ]:
test_results = []
test_predictions = {}

for name, pipe in best_estimators.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    test_predictions[name] = pred

    test_results.append({
        'modelo': name,
        'MAE_€': mean_absolute_error(y_test, pred),
        'RMSE_€': mean_squared_error(y_test, pred) ** 0.5,
        'R2': r2_score(y_test, pred),
        'sMAPE_%': smape(y_test.values, pred),
    })

test_results_df = pd.DataFrame(test_results).sort_values('MAE_€').reset_index(drop=True)
display(test_results_df)


## 15. Comparación con baselines ingenuos

Un modelo de ML solo aporta valor si supera claramente a estrategias triviales:

- **Baseline "día anterior"**: predecir que mañana se facturará lo mismo que el último día conocido.
- **Baseline "estacional"**: predecir que se facturará lo mismo que el mismo día de la semana
  anterior (usando la fecha real, no la posición de fila — el restaurante no abre todos los días).

In [ ]:
fact_por_fecha = df.set_index('fecha')['facturacion']

# Baseline día anterior: ya lo tenemos calculado y alineado con HORIZON
baseline_naive = df_model['facturacion_1d_antes'].iloc[split:].values
mask_b = ~np.isnan(baseline_naive)

# Baseline estacional: mismo día de la semana, 7 días naturales antes (por fecha real)
fechas_semana_previa = fechas_test - pd.Timedelta(days=7)
baseline_seasonal = fechas_semana_previa.map(fact_por_fecha).values
mask_s = ~np.isnan(baseline_seasonal)

baselines_df = pd.DataFrame([
    {
        'modelo': 'Baseline (día anterior)',
        'MAE_€': mean_absolute_error(y_test[mask_b], baseline_naive[mask_b]),
        'RMSE_€': mean_squared_error(y_test[mask_b], baseline_naive[mask_b]) ** 0.5,
        'R2': r2_score(y_test[mask_b], baseline_naive[mask_b]),
        'sMAPE_%': smape(y_test[mask_b].values, baseline_naive[mask_b]),
    },
    {
        'modelo': 'Baseline (semana anterior)',
        'MAE_€': mean_absolute_error(y_test[mask_s], baseline_seasonal[mask_s]),
        'RMSE_€': mean_squared_error(y_test[mask_s], baseline_seasonal[mask_s]) ** 0.5,
        'R2': r2_score(y_test[mask_s], baseline_seasonal[mask_s]),
        'sMAPE_%': smape(y_test[mask_s].values, baseline_seasonal[mask_s]),
    },
])

comparison_df = pd.concat([test_results_df, baselines_df], ignore_index=True).sort_values('MAE_€').reset_index(drop=True)
display(comparison_df)


## 16. Ranking final

Ranking combinado por MAE, RMSE y R² (solo modelos de ML, sin baselines).

In [ ]:
ranking = test_results_df.copy()
ranking['rank_MAE'] = ranking['MAE_€'].rank(method='min')
ranking['rank_RMSE'] = ranking['RMSE_€'].rank(method='min')
ranking['rank_R2'] = ranking['R2'].rank(method='min', ascending=False)
ranking['score_ranking'] = ranking['rank_MAE'] + ranking['rank_RMSE'] + ranking['rank_R2']
ranking = ranking.sort_values(['score_ranking', 'MAE_€']).reset_index(drop=True)

best_model_name = ranking.iloc[0]['modelo']
print('Mejor modelo:', best_model_name)
display(ranking)


## 17. ¿El mejor modelo supera a los baselines?

In [ ]:
best_row = test_results_df.loc[test_results_df['modelo'] == best_model_name].iloc[0]
baseline_naive_mae = baselines_df.loc[baselines_df['modelo'] == 'Baseline (día anterior)', 'MAE_€'].iloc[0]
baseline_seasonal_mae = baselines_df.loc[baselines_df['modelo'] == 'Baseline (semana anterior)', 'MAE_€'].iloc[0]

for label, baseline_mae in [('día anterior', baseline_naive_mae), ('semana anterior', baseline_seasonal_mae)]:
    if best_row['MAE_€'] < baseline_mae:
        mejora = (baseline_mae - best_row['MAE_€']) / baseline_mae * 100
        print(f"Frente al baseline de {label}: el modelo mejora el MAE en un {mejora:.1f}%.")
    else:
        empeora = (best_row['MAE_€'] - baseline_mae) / baseline_mae * 100
        print(f"ATENCIÓN: frente al baseline de {label}, el modelo NO mejora (MAE un {empeora:.1f}% peor).")


## 18. MAE relativo a la facturación media (interpretación de negocio)

No existe un MAE universalmente "bueno": depende del nivel de facturación. Umbrales orientativos
(no son reglas estadísticas formales):

- < 10% → muy bueno
- 10–20% → razonable / bueno
- 20–30% → mejorable
- \> 30% → predicción débil

In [ ]:
mean_revenue_test = y_test.mean()
relative_mae = best_row['MAE_€'] / mean_revenue_test * 100

print(f"Facturación media diaria en test: {mean_revenue_test:.2f} €")
print(f"MAE del mejor modelo ({best_model_name}): {best_row['MAE_€']:.2f} €")
print(f"MAE relativo: {relative_mae:.2f}%")

if relative_mae < 10:
    print("Interpretación orientativa: MUY BUENO")
elif relative_mae < 20:
    print("Interpretación orientativa: BUENO / RAZONABLE")
elif relative_mae < 30:
    print("Interpretación orientativa: MEJORABLE")
else:
    print("Interpretación orientativa: DÉBIL")


## 19. Gráficos — MAE y RMSE por modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = test_results_df.sort_values('MAE_€')
axes[0].bar(plot_df['modelo'], plot_df['MAE_€'], color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_title('MAE en test final')
axes[0].set_ylabel('MAE (€)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(axis='y', alpha=0.3)

plot_df2 = test_results_df.sort_values('RMSE_€')
axes[1].bar(plot_df2['modelo'], plot_df2['RMSE_€'], color='salmon', alpha=0.85, edgecolor='white')
axes[1].set_title('RMSE en test final')
axes[1].set_ylabel('RMSE (€)')
axes[1].tick_params(axis='x', rotation=25)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'mae_rmse_por_modelo.png', dpi=150, bbox_inches='tight')
plt.show()


## 20. Facturación real vs. predicciones (test)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(fechas_test, y_test.values, label='Real', linewidth=3, color='black')

for name, pred in test_predictions.items():
    linewidth = 2.5 if name == best_model_name else 1.2
    alpha = 1.0 if name == best_model_name else 0.6
    plt.plot(fechas_test, pred, label=name, linewidth=linewidth, alpha=alpha)

plt.xlabel('Fecha')
plt.ylabel('Facturación (€)')
plt.title('Facturación real vs. predicciones — test final')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'real_vs_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()


## 21. Análisis de residuos del mejor modelo

In [ ]:
best_pred = test_predictions[best_model_name]

errores = pd.DataFrame({
    'fecha': fechas_test.values,
    'facturacion_real': y_test.values,
    'facturacion_predicha': best_pred,
})
errores['error'] = errores['facturacion_real'] - errores['facturacion_predicha']
errores['error_abs'] = errores['error'].abs()
errores['error_pct'] = np.where(
    errores['facturacion_real'] != 0,
    errores['error_abs'] / errores['facturacion_real'] * 100,
    np.nan,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(errores['facturacion_real'], errores['facturacion_predicha'], alpha=0.7, color='steelblue')
lims = [min(errores['facturacion_real'].min(), errores['facturacion_predicha'].min()),
        max(errores['facturacion_real'].max(), errores['facturacion_predicha'].max())]
axes[0].plot(lims, lims, color='grey', linestyle='--', linewidth=1)
axes[0].set_xlabel('Facturación real (€)')
axes[0].set_ylabel('Facturación predicha (€)')
axes[0].set_title(f'{best_model_name}: real vs. predicho')
axes[0].grid(alpha=0.3)

axes[1].hist(errores['error'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Error (real - predicho) (€)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de residuos')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'residuos_mejor_modelo.png', dpi=150, bbox_inches='tight')
plt.show()

print('Peores 15 predicciones:')
display(errores.sort_values('error_abs', ascending=False).head(15))


## 22. Importancia de variables

Se muestran dos vistas: la importancia nativa (si el modelo es de árboles) y la
**importancia por permutación** sobre el test (más fiable, funciona para cualquier modelo,
mide cuánto empeora el error al barajar cada variable).

In [ ]:
best_pipe = best_estimators[best_model_name]
best_pipe.fit(X_train, y_train)
fitted_model = best_pipe.named_steps['model']
feature_names_out = best_pipe.named_steps['preprocessor'].get_feature_names_out()

if hasattr(fitted_model, 'feature_importances_'):
    native_imp = (
        pd.DataFrame({'variable': feature_names_out, 'importancia': fitted_model.feature_importances_})
        .sort_values('importancia', ascending=False)
        .head(20)
    )
    print(f'Importancia nativa — {best_model_name}:')
    display(native_imp)
else:
    print(f'{best_model_name} no ofrece feature_importances_ nativas (modelo lineal); '
          'ver coeficientes e importancia por permutación.')

perm = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=30,
    random_state=RANDOM_STATE, scoring='neg_mean_absolute_error', n_jobs=-1,
)
perm_imp = (
    pd.DataFrame({
        'variable': X_test.columns,
        'importancia_permutacion': perm.importances_mean,
        'std': perm.importances_std,
    })
    .sort_values('importancia_permutacion', ascending=False)
    .head(20)
)

print('\nImportancia por permutación (impacto en MAE al barajar la variable, en €):')
display(perm_imp)

plt.figure(figsize=(10, 8))
plot_perm = perm_imp.sort_values('importancia_permutacion')
plt.barh(plot_perm['variable'], plot_perm['importancia_permutacion'],
         xerr=plot_perm['std'], color='steelblue', alpha=0.85)
plt.xlabel('Aumento del MAE al barajar la variable (€)')
plt.title(f'Importancia por permutación — {best_model_name}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'importancia_permutacion.png', dpi=150, bbox_inches='tight')
plt.show()


## 23. Guardar resultados y modelo

In [ ]:
cv_summary.to_csv(RESULTS_DIR / 'cv_resultados_por_modelo.csv')
fold_results.to_csv(RESULTS_DIR / 'cv_fold_a_fold.csv', index=False)
search_summary_df.to_csv(RESULTS_DIR / 'busqueda_hiperparametros.csv', index=False)
comparison_df.to_csv(RESULTS_DIR / 'test_vs_baselines.csv', index=False)
errores.to_csv(RESULTS_DIR / 'predicciones_mejor_modelo.csv', index=False)
perm_imp.to_csv(RESULTS_DIR / 'importancia_permutacion.csv', index=False)

joblib.dump(best_pipe, MODELS_DIR / 'mejor_modelo_facturacion_diaria.joblib')

print('Resultados guardados en:', RESULTS_DIR)
print('Modelo guardado en     :', MODELS_DIR / 'mejor_modelo_facturacion_diaria.joblib')


## 24. Conclusión automática

In [ ]:
print('=' * 72)
print('CONCLUSIÓN')
print('=' * 72)
print(f'Horizonte de predicción : {HORIZON} día(s)')
print(f'Mejor modelo (test)     : {best_model_name}')
print(f"MAE                     : {best_row['MAE_€']:.2f} €")
print(f"RMSE                    : {best_row['RMSE_€']:.2f} €")
print(f"R²                      : {best_row['R2']:.4f}")
print(f"sMAPE                   : {best_row['sMAPE_%']:.2f}%")
print(f'MAE relativo            : {relative_mae:.2f}%')

print('\nLimitaciones a tener en cuenta en el TFM:')
print('- Solo se dispone de un ciclo anual completo: no se puede confirmar ni descartar')
print('  estacionalidad anual real (solo se ha visto un verano, una Navidad...).')
print('- El dataset es pequeño para estándares de ML (~200 días útiles tras construir el')
print('  historial), lo que limita la complejidad de modelo que se puede ajustar con garantías.')
print('- Los resultados corresponden a un único restaurante; no se pueden generalizar sin más.')
print('- Aumentar HORIZON reduce la información disponible (menos historial reciente) y por')
print('  tanto se espera que el error crezca cuanto mayor sea la antelación exigida.')
